## Basic Healthcare Provision Fund Machine Learning Accountability System
A complete data engineering and machine learning pipeline that transforms raw government disbursement records and 8,309 facility PDFs into predictive accountability intelligence — built to hold public health funding accountable at national scale.

**Data Extraction & Validation**

This step is the foundation. We read three different document types — Excel files, a newspaper PDF, and a 231-page government PDF, and turn them into clean, structured tables that every other step can use reliably.

In [25]:
!pip install pdfplumber

In [14]:
# Import required libraries
import os
import re
import pandas as pd
import pdfplumber         # specialist library for reading tables from pdf files
from pathlib import Path  # This point to the folder that hold all the 4 government source documents

BASE = Path(r"C:\Users\hp\Basic_healthcare_modelling_project")

# Individual file paths
Q1_file               = BASE / "Q1_disbursements.xlsx"           # Q1 Excel file
Q2_file               = BASE / "Q2_disbursements.xlsx"           # Q2 Excel file
Q3_file               = BASE / "Q3_disbursements.pdf"            # news paper pdf with Q3 table
healthcare_facilities = BASE / "Basic_healthcare_facilities.pdf" # 231-page health facility list pdf  

In [15]:
# Create output folder
OUT_DIR = BASE / "output"
os.makedirs(OUT_DIR, exist_ok=True)

In [16]:
# Verify all source files exist 
for f in [Q1_file, Q2_file, Q3_file, healthcare_facilities]:
    status = "OK" if f.exists() else "NOT FOUND - check file name"
    print(f"{f.name}: {status}")

Q1_disbursements.xlsx: OK
Q2_disbursements.xlsx: OK
Q3_disbursements.pdf: OK
Basic_healthcare_facilities.pdf: OK


In [17]:
# Read quarterly Excel file
def extract_excel_disbursements(filepath, quarter_label):
    raw = pd.read_excel(filepath, header=None, engine="openpyxl")

    # Search every row untill we find one containing the word "STATE"
    header_row_idx = None
    for i, row in raw.iterrows():
        if any(str(v).strip().upper() in ("STATE", "STATES", "S/N") for v in row):
            header_row_idx = i
            break

    # Re-read the file properly, telling pandas which row is the header
    df = pd.read_excel(filepath, header=header_row_idx, engine="openpyxl")

    # Drop columns where every cell is empty
    df.dropna(axis=1, how="all", inplace=True)

    #Remove rows that are totals/subtotals, not actual state data
    mask_total = df.apply(lambda row: row.astype(str).str.upper().str.contains("TOTAL").any(), axis=1)

    # Keep rows that are not totals
    df = df[~mask_total].copy()

    # Identify which column is which, using keyword matching
    col_map = {}
    for col in df.columns:
        cu = col.upper()    # Compare in upper case so "state" == "STATE"
        if col_map.get("state") is None and "STATE" in cu:
            col_map["state"] = col      # Store the actual column name
        elif "NPHCDA" in cu:
            col_map["nphcda"] = col
        elif "NHIA" in cu and "GATEWAY" in cu:
            col_map["nhia_total"] = col
        elif "NHIA" in cu and "EQUAL" in cu:
            col_map["nhia_equal"] = col
        elif "NHIA" in cu and "POOR" in cu:
            col_map["nhia_poor"] = col
        elif "NCDC" in cu:
            col_map["ncdc"] = col
        elif col_map.get("grand_total") is None and "TOTAL" in cu:
            col_map["grand_total"] = col

    # Keep only the columns we identified and rename them to clean names
    df_clean = df[list(col_map.values())].copy()
    df_clean.columns = list(col_map.keys())

    # Clean state names: remove leading serial numbers like "1. Abia" -> "ABIA"
    df_clean["state"] = (
    df_clean["state"]
    .astype(str)
    .str.replace(r"^\d+\.?\s*", "", regex=True) # strip "1." at the start
    .str.strip()  # remove leading/trailing spaces
    .str.upper()  # make everything uppercase for consistency
)

    # Convert the money column from strings to actual numbers
    money_cols = [c for c in df_clean.columns if c != "state"]
    for col in money_cols:
        df_clean[col] = (
            df_clean[col]
            .astype(str)
            .str.replace(r"[₦,\s]", "", regex=True)  # remove ₦ symbol and commas
            .str.replace(r"[^\d.]", "", regex=True)  # keep only digit and "."
            .replace("", "0")  # empty string becomes 0
            .astype(float)     # finally convert to a real decimal number
    )

    df_clean["quarter"] = quarter_label  # add a column marking which quarter
    return df_clean

# call all three extraction functions using the paths defined at the top
q1 = extract_excel_disbursements(Q1_file, "Q1")
q2 = extract_excel_disbursements(Q2_file, "Q2")

**READING THE Q3 DISBURSEMENT NEWS PAPER PDF**

In [22]:
# The Daily Trust newspaper PDF uses a custom embedded font with no Unicode
# mapping. When pdfplumber reads it, all text comes back as (cid:XX) codes
# instead of readable characters — a known limitation of some newspaper PDFs.
# Because automated extraction is impossible, the 37-state table was read
# manually from the PDF and entered here directly.
# Every row was then cross-checked: nphcda + ncdc + emtc + nhia == total_alloc.
# Rivers State NCDC was corrected from a misread ₦18,844,436.92 → ₦7,188,436.92
# using the column-total method (subtracting all other states from the grand total).

    
# filepath is accepted as an argument to keep the interface consistent with Q1/Q2 functions
def extract_q3_from_pdf(filepath):
    print(f"\n[Q3] Loading verified hardcoded data (newspaper PDF uses unreadable font encoding)")

     # Each dict is one state row with 5 values:
    #   nphcda = National Primary Health Care Development Agency gateway
    #   ncdc   = Nigeria Centre for Disease Control gateway
    #   emtc   = Emergency Medical Treatment Committee gateway
    #   nhia   = National Health Insurance Authority gateway
    #   total_alloc = consolidated total (should equal nphcda+ncdc+emtc+nhia)

    q3_rows = [
        {"state":"ABIA",        "nphcda":278153102.50, "ncdc":7300987.92,  "emtc":0.00,        "nhia":294901370.06, "total_alloc":580355460.48},
        {"state":"ADAMAWA",     "nphcda":341644571.54, "ncdc":9908397.42,  "emtc":0.00,        "nhia":450815947.36, "total_alloc":802368916.32},
        {"state":"AKWA IBOM",   "nphcda":497349840.88, "ncdc":7986719.79,  "emtc":0.00,        "nhia":286475308.90, "total_alloc":791811869.56},
        {"state":"ANAMBRA",     "nphcda":492814735.95, "ncdc":7487097.19,  "emtc":6477472.98,  "nhia":271906689.91, "total_alloc":778685996.03},
        {"state":"BAUCHI",      "nphcda":320480748.53, "ncdc":13934926.83, "emtc":8004033.30,  "nhia":540667006.83, "total_alloc":883086715.49},
        {"state":"BAYELSA",     "nphcda":158728672.62, "ncdc":5073418.85,  "emtc":6882705.36,  "nhia":252352473.63, "total_alloc":423037270.46},
        {"state":"BENUE",       "nphcda":417229653.75, "ncdc":7170631.37,  "emtc":0.00,        "nhia":344318874.09, "total_alloc":768719159.20},
        {"state":"BORNO",       "nphcda":471650912.93, "ncdc":11268037.52, "emtc":0.00,        "nhia":495176833.25, "total_alloc":978095783.70},
        {"state":"CROSS RIVER", "nphcda":291758417.29, "ncdc":5581427.58,  "emtc":0.00,        "nhia":317560274.23, "total_alloc":614900119.11},
        {"state":"DELTA",       "nphcda":408159443.88, "ncdc":6698318.60,  "emtc":0.00,        "nhia":237399369.45, "total_alloc":652257131.93},
        {"state":"EBONYI",      "nphcda":258500981.12, "ncdc":7017042.26,  "emtc":7498880.62,  "nhia":380357560.18, "total_alloc":653374464.18},
        {"state":"EDO",         "nphcda":290246715.65, "ncdc":12852650.70, "emtc":0.00,        "nhia":252339653.94, "total_alloc":555439020.29},
        {"state":"EKITI",       "nphcda":267571190.99, "ncdc":5010834.18,  "emtc":6421961.70,  "nhia":279831057.06, "total_alloc":558835043.93},
        {"state":"ENUGU",       "nphcda":394554129.09, "ncdc":7611455.53,  "emtc":0.00,        "nhia":389089476.96, "total_alloc":791255061.57},
        {"state":"GOMBE",       "nphcda":172333987.42, "ncdc":5674626.66,  "emtc":12178481.88, "nhia":372707227.47, "total_alloc":562894323.43},
        {"state":"IMO",         "nphcda":461069001.42, "ncdc":9146607.19,  "emtc":0.00,        "nhia":295362796.95, "total_alloc":765578405.56},
        {"state":"JIGAWA",      "nphcda":435370073.47, "ncdc":6790949.29,  "emtc":0.00,        "nhia":630551604.93, "total_alloc":1072712627.70},
        {"state":"KADUNA",      "nphcda":385483919.22, "ncdc":5994327.29,  "emtc":6421961.70,  "nhia":465614690.03, "total_alloc":863514898.24},
        {"state":"KANO",        "nphcda":731663595.70, "ncdc":7066912.75,  "emtc":6421961.70,  "nhia":757374287.15, "total_alloc":1502526757.31},
        {"state":"KATSINA",     "nphcda":544212591.84, "ncdc":9940747.58,  "emtc":0.00,        "nhia":587565754.68, "total_alloc":1141719094.11},
        {"state":"KEBBI",       "nphcda":340132869.90, "ncdc":8818069.22,  "emtc":0.00,        "nhia":393421421.10, "total_alloc":742372360.22},
        {"state":"KOGI",        "nphcda":361296692.92, "ncdc":7456699.09,  "emtc":0.00,        "nhia":296815036.57, "total_alloc":665568428.57},
        {"state":"KWARA",       "nphcda":291758417.29, "ncdc":9461311.28,  "emtc":0.00,        "nhia":261946864.80, "total_alloc":563166593.37},
        {"state":"LAGOS",       "nphcda":370366902.78, "ncdc":5903895.06,  "emtc":6421961.70,  "nhia":254493288.68, "total_alloc":637186048.23},
        {"state":"NASARAWA",    "nphcda":222220141.67, "ncdc":5803198.44,  "emtc":0.00,        "nhia":321007407.31, "total_alloc":549030747.41},
        {"state":"NIGER",       "nphcda":414206250.46, "ncdc":8831169.49,  "emtc":0.00,        "nhia":500873693.28, "total_alloc":923911113.23},
        {"state":"OGUN",        "nphcda":356761587.98, "ncdc":6169877.76,  "emtc":7448920.46,  "nhia":253614023.62, "total_alloc":623994409.82},
        {"state":"ONDO",        "nphcda":306875433.73, "ncdc":5753163.78,  "emtc":0.00,        "nhia":258144156.38, "total_alloc":570772753.90},
        {"state":"OSUN",        "nphcda":500373244.17, "ncdc":12499292.48, "emtc":6421961.70,  "nhia":239870043.07, "total_alloc":759164541.41},
        {"state":"OYO",         "nphcda":530607277.05, "ncdc":5594814.83,  "emtc":0.00,        "nhia":265585215.40, "total_alloc":801787307.28},
        {"state":"PLATEAU",     "nphcda":312922240.31, "ncdc":5778760.01,  "emtc":6421961.70,  "nhia":381101515.76, "total_alloc":706224477.78},
        {"state":"RIVERS",      "nphcda":482232824.44, "ncdc":7188436.92,  "emtc":6821642.95,  "nhia":329477103.71, "total_alloc":825720008.01},
        {"state":"SOKOTO",      "nphcda":368855201.14, "ncdc":11067558.70, "emtc":0.00,        "nhia":572129714.43, "total_alloc":952052474.26},
        {"state":"TARABA",      "nphcda":253965876.19, "ncdc":8312665.85,  "emtc":6421961.70,  "nhia":417111622.94, "total_alloc":685812126.68},
        {"state":"YOBE",        "nphcda":269082892.63, "ncdc":10440086.33, "emtc":16613833.49, "nhia":383607541.93, "total_alloc":679744354.38},
        {"state":"ZAMFARA",     "nphcda":222220141.67, "ncdc":9792551.47,  "emtc":0.00,        "nhia":490256731.95, "total_alloc":722269425.08},
        {"state":"FCT",         "nphcda":93725501.93,  "ncdc":7536327.95,  "emtc":6421961.70,  "nhia":291353598.42, "total_alloc":399037390.00},
    ]

     # Convert the list of dicts into a DataFrame 
    df = pd.DataFrame(q3_rows)
    df["state"]   = df["state"].str.upper().str.strip()
    df["quarter"] = "Q3"

    # Cross-check every row: the 4 gateway columns must sum to total_alloc
    df["_check"] = df["nphcda"] + df["ncdc"] + df["emtc"] + df["nhia"]
    bad = df[abs(df["_check"] - df["total_alloc"]) > 1.0]
    if not bad.empty:
        print("  WARNING — rows don't balance:", bad[["state", "_check", "total_alloc"]])
    else:
        print(f"  ✓ All {len(df)} rows balance.")

    df.drop(columns=["_check"], inplace=True)   # remove the temporary check column
    print(f"  Rows: {len(df)} | Total: ₦{df['total_alloc'].sum():,.2f}")
    return df

# Call the function
q3 = extract_q3_from_pdf(Q3_file)


[Q3] Loading verified hardcoded data (newspaper PDF uses unreadable font encoding)
  ✓ All 37 rows balance.
  Rows: 37 | Total: ₦27,548,982,678.23


In [25]:
# Maps what the pdf says to what our code understands.
# The pdf has 4 possible values in the "Functionality Status" column
def parse_functionality(text):
    t = str(text).upper().strip()
    if "FUNCTIONAL L2" in t:     # best status: fully functional, level 2
        return "L2"
    if "FUNCTIONAL L1" in t:     # good status: functional, level 1
        return "L1"
    if "PARTIALLY" in t:         # partially working
        return "PARTIAL"
    if "NON" in t and "FUNCTIONAL" in t:  # broken/not working
        return "NON-FUNCTIONAL"
    if "FUNCTIONAL" in t:     # bare "FUNCTIONAL" without a level = assume L1
        return "L1"
    return None   # unrecognized text - skip this row

def extract_facilities_from_pdf(filepath):
    records = []   # will hold every facility row we extract
    current_state = ""   # tracks the current the state (carried forward across rows)
    current_lga = ""   # tracks the current LGA (same reason)

    with pdfplumber.open(filepath) as pdf:
        for page_num, page in enumerate(pdf.pages, start=1):
            # extract_tables() reads the page's table structure.
            # pdf_plumber identifies cells by their bounding boxes.
            tables = page.extract_tables()
            if not tables:
                continue   # skip pages with no table (cover page, index, etc.)

            for table in tables:
                for row in table:
                    if not row or len(row) < 5:
                        continue   # skip empty or incomplete rows

                    # Normalize each cell: upper case, strip whitespace
                    cells = [str(c).replace("\n", " ").strip().upper()
                        if c else "" for c in row]

                    # The first column is the serial number.
                    # Skip rows where it's not a number (header rows, notes)
                    if not re.match(r"^\d+$", cells[0]):
                        continue

                    # Table columns are: [SN, STATE, LGA, WARD, FACILITY, STATUS]
                    state_cell    = cells[1] # column index 1
                    lga_cell      = cells[2] # column index 2
                    ward_cell     = cells[3] # column index 3
                    facility_cell = cells[4] # column index 4
                    status_cell   = cells[5] if len(cells) > 5 else ""

                    # The pdf uses "merged cells" for state/LGA
                    # when 10 facilities share the same LGA, the LGA name only appears in the first row. We carry it forward
                    if state_cell:
                        current_state = state_cell # update when not blank
                    if lga_cell:
                        current_lga = lga_cell

                    # parse the status text into a clean category label
                    status = parse_functionality(status_cell)
                    if not status:
                        continue  # skip if we can't recognize the status

                    # Only save rows where we have all three required pieces
                    if current_state and facility_cell:
                        records.append({
                            "state":     current_state,
                            "lga":       current_lga,
                            "ward":      ward_cell,
                            "facility":  facility_cell,
                            "status":    status,
                    })
    # Convert the list of dictionaries into a DataFrame 
    df = pd.DataFrame(records)
    return df

facilities = extract_facilities_from_pdf(healthcare_facilities)

print(facilities)

        state        lga             ward                          facility  \
0        ABIA  ABA NORTH   ARIARIA MARKET  UMUNNE-ATO PRIMARY HEALTH CENTRE   
1        ABIA  ABA NORTH           EZIAMA      EZIAMA PRIMARY HEALTH CENTRE   
2        ABIA  ABA NORTH  INDUSTRIAL AREA  INDUSTRIAL PRIMARY HEALTH CENTRE   
3        ABIA  ABA NORTH          OGBOR I     OGBOR 1 PRIMARY HEALTH CENTRE   
4        ABIA  ABA NORTH         OGBOR II                    OGBOR 2 CLINIC   
...       ...        ...              ...                               ...   
8304  ZAMFARA      ZURMI       KWASHABAWA  KWASHABAWA PRIMARY HEALTH CENTRE   
8305  ZAMFARA      ZURMI          MASHEMA     MASHEMA PRIMARY HEALTH CENTRE   
8306  ZAMFARA      ZURMI         RUKUDAWA       TSANU PRIMARY HEALTH CENTRE   
8307  ZAMFARA      ZURMI   YAN BUKI/DUTSI             YANBUKI HEALTH CENTRE   
8308  ZAMFARA      ZURMI            ZURMI       ZURMI PRIMARY HEALTH CENTRE   

       status  
0          L1  
1          L1  
2  

In [26]:
# Save 4 outputs
q1.to_csv(OUT_DIR / "q1_disbursements.csv", index=False)  # Q1 Excel → CSV
q2.to_csv(OUT_DIR / "q2_disbursements.csv", index=False)  # Q2 Excel → CSV
q3.to_csv(OUT_DIR / "q3_disbursements.csv", index=False)  # Q3 hardcoded dict → CSV
facilities.to_csv(OUT_DIR / "facilities.csv",  index=False)  # 8,309 PHCs → CSV
print("All 4 files saved to:", OUT_DIR)

All 4 files saved to: C:\Users\hp\Basic_healthcare_modelling_project\output
